# EDA: OSMI Mental Health in Tech Survey (2014)

**Goal:** predict liklihood for an individual to seek `treatment` for mental health from workplace and demographic survey features.

Order of operations: scope candidate features first, then only run quality/missingness checks on columns that are actually in scope.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
df_raw = pd.read_csv('../data/raw/survey.csv')

In [2]:
df = df_raw.copy()
df.shape

(1259, 27)

## 1. Scope candidate features 

Before checking data quality on anything, decide which columns could plausibly be predictors. Checking dtype and unique values first to catch columns that are structurally unusable.

In [3]:
df.columns = df.columns.str.lower()
df.head()

,timestamp,age,gender,country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,No,Yes,Yes,Not sure,No,Yes,Yes,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,No,No,Don't know,No,Don't know,Don't know,Don't know,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,No,Yes,No,No,No,No,Don't know,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,No,Yes,No,Yes,No,No,No,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,Yes,Yes,Yes,No,Don't know,Don't know,Don't know,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


**Excluded before any further analysis:**
- `timestamp` — 1,246 unique values across 1,259 rows. This is effectively a row identifier from when the form was submitted and is not useful as a predictor.
- `comments` — free text. Out of scope for a structured classifier.
- `treatment`- this is going to be our target

In [4]:
excluded_cols = ['timestamp', 'comments']
df = df.drop(columns=excluded_cols)

In [5]:
df.info()
df.nunique().sort_values()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 25 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   age                        1259 non-null   int64 
 1   gender                     1259 non-null   object
 2   country                    1259 non-null   object
 3   state                      744 non-null    object
 4   self_employed              1241 non-null   object
 5   family_history             1259 non-null   object
 6   treatment                  1259 non-null   object
 7   work_interfere             995 non-null    object
 8   no_employees               1259 non-null   object
 9   remote_work                1259 non-null   object
 10  tech_company               1259 non-null   object
 11  benefits                   1259 non-null   object
 12  care_options               1259 non-null   object
 13  wellness_program           1259 non-null   object
 14  seek_hel

obs_consequence               2
self_employed                 2
family_history                2
treatment                     2
remote_work                   2
tech_company                  2
phys_health_interview         3
mental_health_interview       3
supervisor                    3
coworkers                     3
phys_health_consequence       3
mental_health_consequence     3
anonymity                     3
seek_help                     3
care_options                  3
mental_vs_physical            3
benefits                      3
wellness_program              3
work_interfere                4
leave                         5
no_employees                  6
state                        45
country                      48
gender                       49
age                          53
dtype: int64

In [6]:
df['age'].value_counts().sort_index()

age
-1726            1
-29              1
-1               1
 5               1
 8               1
 11              1
 18              7
 19              9
 20              6
 21             16
 22             21
 23             51
 24             46
 25             61
 26             75
 27             71
 28             68
 29             85
 30             63
 31             67
 32             82
 33             70
 34             65
 35             55
 36             37
 37             43
 38             39
 39             33
 40             33
 41             21
 42             20
 43             28
 44             11
 45             12
 46             12
 47              2
 48              6
 49              4
 50              6
 51              5
 53              1
 54              3
 55              3
 56              4
 57              3
 58              1
 60              2
 61              1
 62              1
 65              1
 72              1
 329             1
 9999999

In [7]:
df = df[(df['age'] >= 18) & (df['age'] <= 100)]

In [8]:
df['age'].min(), df['age'].max()

(18, 72)

In [9]:
keep_cols = [c for c in df.columns]
len(keep_cols), keep_cols

(25,
 ['age',
  'gender',
  'country',
  'state',
  'self_employed',
  'family_history',
  'treatment',
  'work_interfere',
  'no_employees',
  'remote_work',
  'tech_company',
  'benefits',
  'care_options',
  'wellness_program',
  'seek_help',
  'anonymity',
  'leave',
  'mental_health_consequence',
  'phys_health_consequence',
  'coworkers',
  'supervisor',
  'mental_health_interview',
  'phys_health_interview',
  'mental_vs_physical',
  'obs_consequence'])

## Check target balance

In [10]:
df['treatment'].value_counts(normalize=True).round(3)

treatment
Yes    0.505
No     0.495
Name: proportion, dtype: float64

Nearly perfectly balanced (50.5% / 49.5%). No resampling needed.

## 3. Missing Values

Only checking the columns still in scope for percentage missing - `comments`/`Timestamp` were already excluded.

In [11]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False)

state                        41.0
work_interfere               20.9
self_employed                 1.4
age                           0.0
seek_help                     0.0
mental_vs_physical            0.0
phys_health_interview         0.0
mental_health_interview       0.0
supervisor                    0.0
coworkers                     0.0
phys_health_consequence       0.0
mental_health_consequence     0.0
leave                         0.0
anonymity                     0.0
care_options                  0.0
wellness_program              0.0
gender                        0.0
benefits                      0.0
tech_company                  0.0
remote_work                   0.0
no_employees                  0.0
treatment                     0.0
family_history                0.0
country                       0.0
obs_consequence               0.0
dtype: float64

Explore whether missing `state` values correspond with U.S.

In [12]:
df['state'].unique()

array(['IL', 'IN', nan, 'TX', 'TN', 'MI', 'OH', 'CA', 'CT', 'MD', 'NY',
       'NC', 'MA', 'IA', 'PA', 'WA', 'WI', 'UT', 'NM', 'OR', 'FL', 'MN',
       'MO', 'AZ', 'CO', 'GA', 'DC', 'NE', 'WV', 'OK', 'KS', 'VA', 'NH',
       'KY', 'AL', 'NV', 'NJ', 'SC', 'VT', 'SD', 'ID', 'MS', 'RI', 'WY',
       'LA', 'ME'], dtype=object)

In [13]:
is_us = df['country'] == 'United States'
state_missing = df['state'].isna()
pd.crosstab(is_us, state_missing, rownames=['is_us'], colnames=['state_missing'])

state_missing,False,True
is_us,,
False,3,502
True,735,11


Most missing values from `state` column are non-US.

In [14]:
excluded_cols.append('state')

## Column: Country

In [15]:
df['country'].value_counts()

country
United States             746
United Kingdom            184
Canada                     72
Germany                    45
Netherlands                27
Ireland                    27
Australia                  21
France                     13
India                      10
New Zealand                 8
Italy                       7
Poland                      7
Sweden                      7
Switzerland                 7
South Africa                6
Belgium                     6
Brazil                      6
Israel                      5
Singapore                   4
Bulgaria                    4
Mexico                      3
Russia                      3
Finland                     3
Austria                     3
Denmark                     2
Greece                      2
Portugal                    2
Colombia                    2
Croatia                     2
Moldova                     1
Georgia                     1
China                       1
Thailand                    1
Cz

In [17]:
(df['country'].value_counts() < 10).sum()

37

**Decisions:**
- `state` - missing mostly for non-US respondents — drop column
- `country`- U.S. is 60% of the data. Will convert to U.S. and non-U.S. categories

## 5. Data quality: Gender

Gender was collected as free text therefore needs to be cleaned

In [18]:
df['gender'].value_counts().head(15)

gender
Male              612
male              204
Female            121
M                 116
female             62
F                  38
m                  34
f                  15
Make                4
Woman               3
Male                3
Female              2
Cis Male            2
Man                 2
Female (trans)      2
Name: count, dtype: int64

In [19]:
df['gender'] = df['gender'].str.lower().str.strip()

In [20]:
df['gender'].value_counts()

gender
male                                              819
female                                            185
m                                                 150
f                                                  53
make                                                4
woman                                               4
cis male                                            3
man                                                 2
female (trans)                                      2
msle                                                1
guy (-ish) ^_^                                      1
male leaning androgynous                            1
trans woman                                         1
queer                                               1
neuter                                              1
agender                                             1
female (cis)                                        1
mail                                                1
malr                 

In [21]:
df['gender'] = df['gender'].apply(
    lambda x: 'female' if ('female' in x or x == 'f') 
    else 'male' if ('male' in x or x == 'm')
    else 'other'
)

In [22]:
df['gender'].value_counts()

gender
male      977
female    244
other      30
Name: count, dtype: int64

**Decisions:**
- `gender` - cleaned values and consolidated into 3 categories

## Checking remaining columns

In [26]:
df = df.drop(columns=excluded_cols, errors='ignore')

In [27]:
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique())
    print()

gender ['female' 'male' 'other']

country ['United States' 'Canada' 'United Kingdom' 'Bulgaria' 'France' 'Portugal'
 'Netherlands' 'Switzerland' 'Poland' 'Australia' 'Germany' 'Russia'
 'Mexico' 'Brazil' 'Slovenia' 'Costa Rica' 'Austria' 'Ireland' 'India'
 'South Africa' 'Italy' 'Sweden' 'Colombia' 'Latvia' 'Romania' 'Belgium'
 'New Zealand' 'Spain' 'Finland' 'Uruguay' 'Israel'
 'Bosnia and Herzegovina' 'Hungary' 'Singapore' 'Japan' 'Nigeria'
 'Croatia' 'Norway' 'Thailand' 'Denmark' 'Greece' 'Moldova' 'Georgia'
 'China' 'Czech Republic' 'Philippines']

self_employed [nan 'Yes' 'No']

family_history ['No' 'Yes']

treatment ['Yes' 'No']

work_interfere ['Often' 'Rarely' 'Never' 'Sometimes' nan]

no_employees ['6-25' 'More than 1000' '26-100' '100-500' '1-5' '500-1000']

remote_work ['No' 'Yes']

tech_company ['Yes' 'No']

benefits ['Yes' "Don't know" 'No']

care_options ['Not sure' 'No' 'Yes']

wellness_program ['No' "Don't know" 'Yes']

seek_help ['Yes' "Don't know" 'No']

anonymity ['Y

In [35]:
df = df.replace({"Don't know": "Unsure", "Not sure": "Unsure", "Maybe": "Unsure"})

In [36]:
for col in ['benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity',
            'mental_vs_physical', 'mental_health_consequence', 'phys_health_consequence',
            'mental_health_interview', 'phys_health_interview', 'coworkers', 'supervisor', 'leave']:
    print(col, df[col].unique())

benefits ['Yes' 'No' 'Unsure']
care_options ['Unsure' 'No' 'Yes']
wellness_program ['Yes' 'No' 'Unsure']
seek_help ['Unsure' 'No' 'Yes']
anonymity ['Yes' 'No' 'Unsure']
mental_vs_physical ['Yes' 'Unsure' 'No']
mental_health_consequence ['No' 'Unsure' 'Yes']
phys_health_consequence ['No' 'Unsure' 'Yes']
mental_health_interview ['No' 'Unsure' 'Yes']
phys_health_interview ['Yes' 'Unsure' 'No']
coworkers ['Yes' 'Some of them' 'No']
supervisor ['Yes' 'Some of them' 'No']
leave ['Very easy' 'Somewhat easy' 'Somewhat difficult' 'Unsure'
 'Very difficult']


## Col: Self Employed

In [39]:
df['self_employed'].isnull().sum()

0

In [30]:
df = df.dropna(subset=['self_employed'])

In [32]:
df['self_employed'].isnull().sum()

0

## Col: Work interfere

In [38]:
df['work_interfere'].isnull().sum()

262

In [40]:
df[df['work_interfere'].isnull()]['treatment'].value_counts()

treatment
No     258
Yes      4
Name: count, dtype: int64

The majority of those who did not seek treatment also skipped the question. Drop the remaining 4 that correspond to those that did seek treatment but did not answer the question to preserve integrity.

In [41]:
drop_mask = df['work_interfere'].isnull() & (df['treatment'] == 'Yes')
df = df[~drop_mask]

In [42]:
df['work_interfere'].isnull().sum()

258

In [44]:
df['work_interfere'] = df['work_interfere'].fillna('Not applicable')

In [46]:
df['work_interfere'].isnull().sum()

0

**Decisions**

- All columns - "Don't know", "Not sure", "Maybe" all changed to "Unsure"
- `work_interfere` - converted NaNs to "Not applicable"

In [47]:
pd.crosstab(df['work_interfere'], df['treatment'])

treatment,No,Yes
work_interfere,,
Never,176,30
Not applicable,258,0
Often,21,117
Rarely,49,121
Sometimes,106,351


In [48]:
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique())
    print()

gender ['male' 'female' 'other']

country ['United States' 'France' 'United Kingdom' 'Canada' 'Portugal'
 'Netherlands' 'Switzerland' 'Poland' 'Australia' 'Germany' 'Russia'
 'Mexico' 'Brazil' 'Slovenia' 'Costa Rica' 'Austria' 'Ireland' 'India'
 'South Africa' 'Italy' 'Bulgaria' 'Sweden' 'Colombia' 'Latvia' 'Romania'
 'Belgium' 'New Zealand' 'Spain' 'Finland' 'Uruguay' 'Israel'
 'Bosnia and Herzegovina' 'Hungary' 'Singapore' 'Japan' 'Nigeria'
 'Croatia' 'Norway' 'Thailand' 'Denmark' 'Greece' 'Moldova' 'Georgia'
 'China' 'Czech Republic' 'Philippines']

self_employed ['Yes' 'No']

family_history ['Yes' 'No']

treatment ['No' 'Yes']

work_interfere ['Sometimes' 'Not applicable' 'Never' 'Often' 'Rarely']

no_employees ['1-5' '6-25' '100-500' '26-100' 'More than 1000' '500-1000']

remote_work ['Yes' 'No']

tech_company ['Yes' 'No']

benefits ['Yes' 'No' 'Unsure']

care_options ['Unsure' 'No' 'Yes']

wellness_program ['Yes' 'No' 'Unsure']

seek_help ['Unsure' 'No' 'Yes']

anonymity ['Yes' '

**Approach for ML**

- Feature set A (baseline) are all cleaned columns and while dropping those that were not able to be encoded.

**Encoding plan:**

Binary (map to 1/0):
- `self_employed`, `family_history`, `treatment` (target), `remote_work`, `tech_company`, `obs_consequence`

Ordinal (has a natural order — map to numbers):
- `work_interfere`: not applicable(0) < never(1) < rarely(2) < sometimes(3) < often(4)
- `leave`: very easy(0) < somewhat easy(1) < unsure(2) < somewhat difficult(3) < very difficult(4)
- `no_employees`: 1-5(0) < 6-25(1) < 26-100(2) < 100-500(3) < 500-1000(4) < more than 1000(5)

Nominal (no order — one-hot encode):
- `gender`, `country`, `benefits`, `care_options`, `wellness_program`, `seek_help`, `anonymity`,
  `mental_vs_physical`, `mental_health_consequence`, `phys_health_consequence`,
  `coworkers`, `supervisor`, `mental_health_interview`, `phys_health_interview`